In [10]:
### write_card.py
import re
from typing import Optional
# 引擎接入调用

%run ./knowledge_engine.ipynb
from knowledge_engine import MarkdownHeaderParser, ParsedHeader


#01-1

# 这是系统的基石：解析协议的纯函数
def parse_title_to_json(md_text: str) -> dict:
    """
    输入: Markdown文本
    输出: 标准化的知识卡片协议
    策略: 代理模式 - 调用 Engine 解析，仅做数据格式转换
    """
    # 1. 实例化 Engine (这里为了演示，实际应由依赖注入传入)
    parser = MarkdownHeaderParser()
    
    # 2. 调用 Engine 的核心功能 (注意：Engine 返回的是 List)
    # 这里只取第一个匹配的标题，或者需要处理多个
    engine_results = parser.parse_header(md_text) 
    
    if not engine_results:
        return {"error": "格式不匹配", "raw": md_text}
    
    # 3. 将 Engine 的对象 转换为 WriteCard 的 JSON 协议 (适配器模式)
    # 取第一个结果为例
    engine_obj: ParsedHeader = engine_results[0]
    
    # 4. 构建 WriteCard 需要的结构 (注意这里的字段映射)
    return {
        "meta": {
            # 注意：这里修正了之前的变量名错误 # 使用清洗后的编码
            "full_code": f"{engine_obj.category}-{engine_obj.subject}-{engine_obj.sequence}",
            "category_code": engine_obj.category, # 对应 AAA
            "subject": engine_obj.subject,        # 对应 BBB (修正了原代码的错误)
            "title": f"{engine_obj.category} {engine_obj.subject} {engine_obj.sequence}" # 或者从 metadata 获取
        },   # 需逻辑映射到 "记忆/专业" 等
        "content": {
            # 关键点：Engine 不提供 Body，这里必须留空或由其他函数处理
            # 原 write_card 的 body 提取逻辑在这里是禁止的
            # 因为 Engine 没有提供这个功能，不能在这里擅自切分字符串
            "body": "" 
        },
        "routing": {}# 此时还不知道要写哪，由后续函数填充
    }
    clean_code = "-".join(parts)
    

# 测试用例
# if __name__ == "__main__":
#     sample = """### 【100-001-Python】装饰器详解
#     - 这是正文内容"""
#     print(parse_title_to_json(sample))
    

# #01-2
from pathlib import Path
import json
from typing import Dict, Any, Optional

def atomic_write_card(file_path: str, card_data: Dict[str, Any], mode: str = 'append') -> Dict[str, Any]:
    """
    原子写入操作：适配 MarkdownHeaderParser (Engine)
    
    注意：此函数假设 card_data 的结构已经符合 Engine 的解析规范。
    它不负责校验逻辑，只负责安全地将数据渲染并写入磁盘。
    原子写入操作：只负责把内容写到指定位置，不管逻辑。
    包含安全校验。
    Args:
        file_path (str): 目标文件路径
        card_data (dict): 卡片数据，必须包含 'header' 和 'content' 键
        mode (str): 'append' 或 'overwrite'
    
    Returns:
        dict: 操作结果
    """
    path = Path(file_path)
    
    # 1. 安全校验与目录创建
    # ----------------------------------------
    # Engine 假设目录存在，WriteCard 负责确保这一点
    if not path.parent.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        
    # 2. 数据渲染 (适配层) 格式化内容
    # ----------------------------------------
    # 关键步骤：将通用的 Python Dict 转换为 Engine 能解析的纯 Markdown 字符串
    # 这里调用 _render_markdown 进行格式化
    try:
        text_to_write = _render_markdown(card_data)
    except Exception as e:
        return {"status": "error", "message": f"渲染失败: {str(e)}"}

    # 3. 原子写入
    # ----------------------------------------
    write_mode = 'a' if mode == 'append' else 'w'
    try:
        with open(path, write_mode, encoding='utf-8') as f:
            # 追加模式下增加分隔符，防止 Engine 解析混乱
            if mode == 'append' and path.exists():
                f.write("\n\n---\n\n")# 分隔符
            f.write(text_to_write)
        
        return {"status": "success", "path": str(path), "written": len(text_to_write)}
    
    except Exception as e:
        return {"status": "error", "message": f"写入失败: {str(e)}"}


    #1-03
def _render_markdown(data: Dict[str, Any]) -> str:
    """
    内部渲染函数：严格遵循 MarkdownHeaderParser 的正则规范
    
    Engine (fileUp.py) 的正则: r'###?\s*【([^】]+)】\s*(.+?)(?:\n|$)'
    解析逻辑：### 【AAA-BBB-CCC】 标题文本
    
    本函数必须生成完全符合上述规则的字符串，否则 Engine 无法读取。
    """
    # --- 第一步：提取 Header 数据 (这是 Engine 的核心) ---
    # Engine 需要 AAA-BBB-CCC 结构
    header = data.get('header', {})
    
    category = header.get('category', '未分类')
    subject = header.get('subject', '未定义')
    sequence = header.get('sequence', '000') # Engine 要求至少3位数字
    
    # 强制格式化序号，防止 Engine 报错
    if not sequence.isdigit() or len(sequence) < 3:
        sequence = sequence.zfill(3)[:10] # 补零并限制长度
    
    # 构建 Engine 能识别的 Code
    # 注意：这里必须使用短横线连接，且不能有空格
    header_code = f"{category}-{subject}-{sequence}"
    
    # 提取标题文本
    # 注意：Engine 的正则捕获的是 【...】后面紧跟的文本 (即 title)
    title_text = header.get('title', '无标题').strip()
    
    # --- 第二步：构建符合 Engine 规范的 Markdown ---
    # 严格按照 Engine 的正则模式拼接
    # 生成：### 【AAA-BBB-CCC】标题文本
    # 注意：中间没有换行，这是 Engine 解析的关键
    markdown_lines = []
    markdown_lines.append(f"### 【{header_code}】 {title_text}")
    
    # --- 第三步：处理正文与元数据 (非侵入式) ---
    # 注意：Engine 只解析 Header，正文部分可以自由扩展
    # 但为了防止污染，我们在正文开始处加一个空行
    content = data.get('content', {})
    body = content.get('body', '').strip()
    
    # 如果有正文，加一个空行与 Header 隔开
    if body:
        markdown_lines.append("")
        markdown_lines.append(body)
    
    # --- 第四步：可选的元数据 (Tags/Sources) ---
    # 注意：这部分不能干扰 Engine 的 Header 解析
    # 所以放在最后
    tags = content.get('tags', [])
    source = content.get('source', '')
    
    if tags or source:
        markdown_lines.append("")
        markdown_lines.append("---") # 视觉分隔符
        if tags:
            markdown_lines.append(f"- **标签**: {', '.join(tags)}")
        if source:
            markdown_lines.append(f"- **来源**: {source}")

    return "\n".join(markdown_lines)




示例配置文件已创建: E:\Briah\config\parser_config.yaml
解析到 2 个标题
{
  "raw_header": "【Python-编程-001】装饰器详解",
  "category": "Python",
  "subject": "编程",
  "sequence": "001",
  "components": {
    "category": "Python",
    "subject": "编程",
    "sequence": "001"
  },
  "card_type": null,
  "metadata": {
    "parsed_at": "2026-03-31T16:54:57.709075",
    "title_text": "装饰器详解",
    "original_code": "Python-编程-001"
  }
}
{
  "raw_header": "【JavaScript-前端-002】ES6新特性",
  "category": "JavaScript",
  "subject": "前端",
  "sequence": "002",
  "components": {
    "category": "JavaScript",
    "subject": "前端",
    "sequence": "002"
  },
  "card_type": null,
  "metadata": {
    "parsed_at": "2026-03-31T16:54:57.709075",
    "title_text": "ES6新特性",
    "original_code": "JavaScript-前端-002"
  }
}
解析结果已保存到: E:\Briah\记忆库_parsed.json

生成的标题: ### 【Python-算法-003-v1】快速排序算法详解
None


In [13]:
# 【CREATE 阶段】：构建符合 Engine 规范的数据测试

def main():
    # 1. 【CREATE 阶段】：构建符合 Engine 规范的数据
    # --------------------------------------------------
    # 注意：这里的结构必须严格对应 write_card.py 中 _render_markdown 的要求
    new_card_data = {
        "header": {
            "category": "Python",
            "subject": "算法",
            "sequence": "003",
            "title": "快速排序算法详解",
            "version": "1"}}
    
print(main())


None


In [8]:
# 测试函数，用来验证 write_card 是否能调用 engine
def test_parse_header():
    parser = MarkdownHeaderParser()
    # 测试内容必须包含符合格式的标题
    content = "### 【Python-编程-001】装饰器详解\n### 【JavaScript-前端-002】ES6新特性"
    result = parser.parse_header(content)
    assert len(result) == 2
    assert result[0].category == "Python"
    assert result[0].subject == "编程"
    assert result[0].sequence == "001"

    
def test_engine_integration():
    """
    这是一个简单的测试函数，用来验证 write_card 是否能调用 engine，先headerparser后尾
    """
    print("--- 🧪 正在测试 knowledge_engine 集成 ---")

    # 2. 实例化工具
    # 这里相当于把解析器这个“模具”拿了出来
    parser = MarkdownHeaderParser()

    # 3. 准备测试数据
    # 模拟一段你可能要写入的 Markdown 内容
    test_md_content = """
### 【Python-编程-001】装饰器详解\n- *脸埋在你胸口*
    """

    # 4. 调用 engine 的功能
    # 假设 engine 有一个方法叫 parse_headers，用来解析头部信息
    try:
        # 这里调用 engine 里的方法，传入测试内容
        result = parser.parse_header(test_md_content)

        # 5. 打印结果
        print("✅ 调用成功！解析结果如下：")
        print(result)

    except Exception as e:
        print(f"❌ 调用失败：{e}")

if __name__ == "__main__":
    # 先测试核心解析功能
    test_parse_header()
    
    # 再测试引擎集成
    test_engine_integration()

--- 🧪 正在测试 knowledge_engine 集成 ---
✅ 调用成功！解析结果如下：
[ParsedHeader(raw_header='【Python-编程-001】装饰器详解', category='Python', subject='编程', sequence='001', version=None, card_type=None, components={'category': 'Python', 'subject': '编程', 'sequence': '001'}, metadata={'parsed_at': '2026-03-31T16:32:06.224159', 'title_text': '装饰器详解', 'original_code': 'Python-编程-001'})]


In [15]:
# # 从文件读取Markdown,并解析为JSON格式          

file_path = "./记忆库.md"
with open(file_path, "r", encoding="utf-8") as f:
    md_text = f.read()
card_data = parse_title_to_json(md_text)
print(card_data)

{'error': '格式不匹配', 'raw': '# 目录\n\n## 办公软件\n### 【Word】快速调整行距 \n问题：Word 行距调整繁琐 \n解决方案：使用快捷键 Ctrl+1（单倍）/Ctrl+2（双倍） \n关键步骤：选中文本 → 按对应快捷键 \n## 编程开发 \n### 【Python】快速运行代码 \n问题：每次运行Python代码需点击按钮 \n解决方案：使用[[VSCode]]快捷键 F5 \n关键步骤：打开代码文件 → 按 F5 运行\n\n## 知识管理\n### 【知识编码】HTTP码式分层编码实操指引 \n问题：知识库知识点无统一标识，检索效率低、易混乱 \n解决方案：按「000/100/200大类码-001子类码-自定义参数码」规则编码，全程复用Doubao.md定义的框架 \n关键步骤： \n1. 定大类：根据知识点类型匹配大类码（000元认知/100口口/200口口/300口口）； \n2. 编子类：在对应大类下按已有知识点数量递增排序（补0至3位，如第1个=001）； \n3. 设参数：按知识点核心特征定义参数码（时长/名称/优先级均可）；\n4. 加前缀：将完整编码作为【】包裹的标题前缀，格式：【XXX-XXX-XXX】知识点标题；\n5. 做检索：直接用「编码+关键词」检索，如「000-001 学习理论」。\n实操示例（可直接复制模板）： \n- 认知类：【000-001-111】000-001-111学习理论 " 技术类：【100-001-Python】Python快速运行代码快捷键 - 职场类：【300-001-P1】职场高效沟通核心技巧 - 兴趣类：【400-001-摄影】摄影构图三分法技巧 "\n检索示例（直接发送以下指令即可）： \n- 帮我检索000-001-111相关内容 \n- 帮我检索000-002-001豆包AI记忆操作的相关内容\n\n## 编码规则示例\n### 【000-001-111】000-001-111学习理论执行框架 \n问题：学习新技能效率低，难以从入门到精通 \n解决方案：遵循「1小时莽+10小时猛+100小时稳」三阶段框架，配合[[000-001-111-脉冲节奏]]与产出导向 \n关键步骤： \n1. 莽（1小时）：跳过前置门槛，完成最小可行案例，确认「我能搞」 \n2.

In [18]:
# 01-4clean_memory_bank.py
# 基于 fileUp.py (Engine) 进行适配，消除功能重叠

# 01-4clean_memory_bank.py (Debug 修复版)

def clean_memory_bank(file_path: str) -> list:
    cards = []
    file_path = Path(file_path)
    
    # --- Debug 1: 检查文件读取 ---
    if not file_path.exists():
        print(f"❌ 错误：文件 {file_path} 不存在")
        return cards
        
    print(f"✅ 文件读取成功，文件大小: {file_path.stat().st_size} 字节")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        markdown_content = f.read()
    
    # --- Debug 2: 检查内容是否包含目标关键词 ---
    print(f"📄 文件前 200 字符预览: {repr(markdown_content[:200])}")
    if "【" not in markdown_content or "###" not in markdown_content:
        print("⚠️ 警告：文件中未检测到 Markdown 标题标记 '###' 或 '【'，请检查文件内容格式。")
        return cards

    parser = MarkdownHeaderParser() 
    
    # --- Debug 3: 检查正则底层匹配 ---
    # 我们先绕过 parse_header，直接看正则能抓到什么
    raw_matches = parser._header_pattern.findall(markdown_content)
    print(f"\n🔍 Engine 底层正则抓取结果 (Raw Matches): {len(raw_matches)} 个")
    for i, match in enumerate(raw_matches):
        print(f"   Match {i+1}: Group1(编码)='{match[0]}', Group2(标题)='{match[1]}'")

    # --- 正常解析流程 ---
    # 即使 raw_matches 有数据，但如果 _parse_header_code 内部校验失败，也会被丢弃
    parsed_headers = parser.parse_header(markdown_content)
    print(f"\n🧠 Engine 最终解析对象数量: {len(parsed_headers)}")

    for header in parsed_headers:
        try:
            card_dict = header.to_json()
            cards.append(card_dict)
        except Exception as e:
            print(f"转换失败: {e}")
            continue

    return cards

if __name__ == "__main__":
    print("🚀 开始清洗任务...")
    all_cards = clean_memory_bank("./记忆库.md")
    
    with open("cleaned_cards.json", "w", encoding="utf-8") as f:
        json.dump(all_cards, f, ensure_ascii=False, indent=2)
        
    print(f"\n✅ 最终结果：成功清洗出 {len(all_cards)} 张卡片")
# def clean_memory_bank(file_path: str) -> list:
#     """
#     清洗记忆库：利用 Engine 的解析能力提取标准卡片
#     逻辑变更：不再手动切分字符串，直接交给 Engine 的正则处理，避免格式冲突。
#     """
#     cards = []
#     file_path = Path(file_path)
    
#     if not file_path.exists():
#         print(f"错误：文件 {file_path} 不存在")
#         return cards

#     # 1. 读取原始内容 (保持原样，不进行任何切分)
#     # ----------------------------------------
#     # Engine 的正则设计就是为了处理多行文本的
#     # 我们不再用 re.split，直接把全文喂给 Engine
#     with open(file_path, 'r', encoding='utf-8') as f:
#         markdown_content = f.read()

#     # 2. 初始化 Engine 解析器
#     # ----------------------------------------
#     # 如果你有 config/ 目录，可以传入配置文件
#     # 这里为了通用性，不传配置（使用默认逻辑）
#     parser = MarkdownHeaderParser() 

#     # 3. 调用 Engine 的核心解析逻辑 (消除重叠)
#     # ----------------------------------------
#     # 原 01-4 想自己写正则切分，现在直接用 Engine 的 parse_header
#     # 这是消除冲突的关键：信任 Engine 的解析能力
#     parsed_headers = parser.parse_header(markdown_content)
    
#     print(f"Engine 检测到 {len(parsed_headers)} 个潜在标题")

#     # 4. 转换为 JSON 格式 (适配前端/存储需求)
#     # ----------------------------------------
#     # Engine 返回的是 ParsedHeader 对象列表
#     # 我们需要转成字典列表 (JSON Serializable)
#     for header in parsed_headers:
#         try:
#             # Engine 提供了标准的 to_json 方法
#             # 这里确保了字段名 100% 符合 Engine 的定义
#             card_dict = header.to_json()
#             cards.append(card_dict)
#         except Exception as e:
#             print(f"警告：卡片转换失败 - {header.raw_header[:20]}... | 错误: {e}")
#             continue

#     return cards

# # --- 执行清洗 ---
# if __name__ == "__main__":
#     # 注意：这里不再需要复杂的切分逻辑
#     all_cards = clean_memory_bank("./记忆库.md")
    
#     # 输出结果
#     output_file = "cleaned_cards.json"
#     with open(output_file, "w", encoding="utf-8") as f:
#         json.dump(all_cards, f, ensure_ascii=False, indent=2)
        
#     print(f"✅ 成功清洗出 {len(all_cards)} 张卡片")
#     print(f"📄 结果已保存到: {output_file}")

🚀 开始清洗任务...
✅ 文件读取成功，文件大小: 47541 字节
📄 文件前 200 字符预览: '# 目录\n\n## 办公软件\n### 【Word】快速调整行距 \n问题：Word 行距调整繁琐 \n解决方案：使用快捷键 Ctrl+1（单倍）/Ctrl+2（双倍） \n关键步骤：选中文本 → 按对应快捷键 \n## 编程开发 \n### 【Python】快速运行代码 \n问题：每次运行Python代码需点击按钮 \n解决方案：使用[[VSCode]]快捷键 F5 \n关键步骤：打开代码文件 → 按 F5 运行\n'

🔍 Engine 底层正则抓取结果 (Raw Matches): 9 个
   Match 1: Group1(编码)='Word', Group2(标题)='快速调整行距 '
   Match 2: Group1(编码)='Python', Group2(标题)='快速运行代码 '
   Match 3: Group1(编码)='知识编码', Group2(标题)='HTTP码式分层编码实操指引 '
   Match 4: Group1(编码)='000-001-111', Group2(标题)='000-001-111学习理论执行框架 '
   Match 5: Group1(编码)='000-001-111', Group2(标题)='000-001-111学习理论标准化执行SOP '
   Match 6: Group1(编码)='000-002-001', Group2(标题)='豆包AI记忆操作实操指引 '
   Match 7: Group1(编码)='000-003 - 像素', Group2(标题)='元认知自学三元技战法核心方法论'
   Match 8: Group1(编码)='002-001-舞厅', Group2(标题)='职场新人“跟对大佬”的核心方法论 '
   Match 9: Group1(编码)='002-002-大圣', Group2(标题)='四大名著与现代治理、商业逻辑核心方法论 '

🧠 Engine 最终解析对象数量: 0

✅ 最终结果：成功清洗出 0 张卡片


In [19]:
# ai_client.py应当集成的基于clean_card.json(write_card module 的初级产品)，做得应答交互
#build_frameworks.py    plugins-aienhanced
"""
知识框架构建器
将cleaned_cards.json转换为结构化框架，用于指导AI思考
"""
import json
from typing import Dict, List, Optional, TypedDict
from pathlib import Path

# 定义类型别名，解决 NameError 问题
CardDict = Dict[str, any]
FrameworkDict = Dict[str, List[Dict[str, any]]]

class FrameworkEntry(TypedDict):
    """框架条目类型定义"""
    code: str
    title: str
    problem_pattern: Optional[str]
    solution_pattern: Optional[str]
    key_steps: List[str]
    example: str

def cards_to_structured_frameworks(file_path: str) -> FrameworkDict:
    """
    将卡片转换为结构化的思维框架
    这是最接近"让AI学会思考方式"的方法
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        cards = json.load(f)
    
    frameworks: FrameworkDict = {
        "learning_methods": [],     # 学习方法框架 (000系列)
        "coding_patterns": [],      # 技术模式框架 (100系列)
        "problem_solving": [],      # 问题解决框架 (200系列)
        "workplace_strategies": [], # 职场策略框架 (300系列)
        "other_strategies": []      # 其他策略框架 (400+系列)
    }
    
    for card in cards:
        # 根据分类码映射到不同的框架类型
        category = card['meta']['category_code']
        
        # 提取问题-解决方案结构
        content = card['content']['body']
        lines = content.split('\n')
        
        problem = None
        solution = None
        steps: List[str] = []
        
        for line in lines:
            line = line.strip()
            if line.startswith('问题：'):
                problem = line.replace('问题：', '').strip()
            elif line.startswith('解决方案：'):
                solution = line.replace('解决方案：', '').strip()
            elif line.startswith('- '):
                steps.append(line[2:].strip())
            elif line.startswith('1. ') and len(line) > 3:
                steps.append(line[3:].strip())
        
        framework_entry: FrameworkEntry = {
            "code": card['meta']['full_code'],
            "title": card['meta']['title'],
            "problem_pattern": problem,
            "solution_pattern": solution,
            "key_steps": steps,
            "example": content[:100] + "..." if len(content) > 100 else content
        }
        
        # 根据分类放入不同的框架组
        if category.startswith('000'):  # 元认知/学习方法
            frameworks['learning_methods'].append(framework_entry)
        elif category.startswith('100'):  # 专业/技术
            frameworks['coding_patterns'].append(framework_entry)
        elif category.startswith('200'):  # 思维/认知
            frameworks['problem_solving'].append(framework_entry)
        elif category.startswith('300'):  # 职场
            frameworks['workplace_strategies'].append(framework_entry)
        else:  # 其他分类
            frameworks['other_strategies'].append(framework_entry)
    
    return frameworks

def create_thinking_prompt_with_frameworks(query: str, frameworks: FrameworkDict) -> str:
    """
    使用框架化的知识指导AI思考
    """
    # 为每个框架类别选择代表性的示例
    example_summary = _build_framework_examples(frameworks)
    
    prompt = f"""# 结构化思考助手

## 可用知识框架
你掌握以下结构化思维框架，可用于分析问题：

{example_summary}

## 当前问题
{query}

## 思考指令
请严格按以下步骤思考：

1. **识别问题类型**：判断问题属于哪个框架类别（学习方法、技术问题、问题解决、职场策略等）
2. **选择适用框架**：从可用框架中选择1-3个最适合的框架
3. **应用框架分析**：使用所选框架的结构（问题模式→解决方案→关键步骤）分析问题
4. **生成具体建议**：提供可执行、可验证的具体建议
5. **注明框架来源**：明确说明使用了哪个知识卡片（编码：XXX-XXX-XXX）

## 输出格式
- **框架选择**：[列出使用的框架编码]
- **分析过程**：[展示如何使用框架分析问题]
- **具体建议**：[按步骤给出可执行建议]
- **预期效果**：[说明实施后的预期效果]

请开始思考..."""
    
    return prompt

def _build_framework_examples(frameworks: FrameworkDict) -> str:
    """构建框架示例摘要"""
    summary_lines = []
    
    for category_name, framework_list in frameworks.items():
        if not framework_list:
            continue
            
        # 获取前两个框架作为示例
        examples = framework_list[:2]
        
        summary_lines.append(f"### {_get_category_label(category_name)} ({len(framework_list)}个模式)")
        
        for i, framework in enumerate(examples, 1):
            summary_lines.append(f"  **{i}. 【{framework['code']}】{framework['title']}**")
            
            if framework['problem_pattern']:
                summary_lines.append(f"     问题模式: {framework['problem_pattern'][:60]}...")
            if framework['solution_pattern']:
                summary_lines.append(f"     解决方案: {framework['solution_pattern'][:60]}...")
            if framework['key_steps']:
                steps_text = '; '.join(framework['key_steps'][:2])
                summary_lines.append(f"     关键步骤: {steps_text[:80]}...")
            
            summary_lines.append("")  # 空行分隔
    
    return "\n".join(summary_lines)

def _get_category_label(category_name: str) -> str:
    """获取分类的中文标签"""
    labels = {
        "learning_methods": "学习方法框架",
        "coding_patterns": "技术模式框架",
        "problem_solving": "问题解决框架",
        "workplace_strategies": "职场策略框架",
        "other_strategies": "其他策略框架"
    }
    return labels.get(category_name, category_name)

def search_relevant_frameworks(query: str, frameworks: FrameworkDict, top_k: int = 3) -> List[FrameworkEntry]:
    """搜索与查询相关的框架"""
    query_lower = query.lower()
    results = []
    
    for category_name, framework_list in frameworks.items():
        for framework in framework_list:
            score = 0
            
            # 标题匹配
            if query_lower in framework['title'].lower():
                score += 3
            
            # 问题模式匹配
            if framework['problem_pattern'] and query_lower in framework['problem_pattern'].lower():
                score += 2
            
            # 解决方案匹配
            if framework['solution_pattern'] and query_lower in framework['solution_pattern'].lower():
                score += 1
            
            if score > 0:
                result = framework.copy()
                result['_score'] = score
                result['_category'] = category_name
                results.append(result)
    
    # 按分数排序
    results.sort(key=lambda x: x['_score'], reverse=True)
    return results[:top_k]

# 使用示例
if __name__ == "__main__":
    # 测试数据路径
    try:
        frameworks = cards_to_structured_frameworks("cleaned_cards.json")
    except FileNotFoundError:
        print("错误: 找不到cleaned_cards.json文件")
        print("请确保文件存在或提供正确的路径")
        exit(1)
    
    # 查看框架摘要
    print("=" * 60)
    print("知识框架摘要")
    print("=" * 60)
    
    for category_name, framework_list in frameworks.items():
        if framework_list:
            print(f"\n{_get_category_label(category_name)}: {len(framework_list)}个框架")
            for framework in framework_list[:3]:  # 显示前三个
                print(f"  - 【{framework['code']}】{framework['title']}")
    
    # 测试搜索功能
    print("\n" + "=" * 60)
    print("框架搜索测试")
    print("=" * 60)
    
    test_queries = ["学习", "Python", "职场", "问题解决"]
    
    for query in test_queries:
        print(f"\n搜索: '{query}'")
        results = search_relevant_frameworks(query, frameworks, top_k=2)
        
        if results:
            for i, result in enumerate(results, 1):
                print(f"  {i}. 【{result['code']}】{result['title']} (分数: {result['_score']})")
        else:
            print("  未找到相关框架")
    
    # 创建指导性Prompt示例
    print("\n" + "=" * 60)
    print("思考提示生成示例")
    print("=" * 60)
    
    user_queries = [
        "我在学习Python装饰器时卡住了，怎么突破？",
        "作为职场新人，如何选择合适的领导？",
        "如何高效学习一个新技能？"
    ]
    
    for i, query in enumerate(user_queries, 1):
        print(f"\n示例 {i}: {query}")
        prompt = create_thinking_prompt_with_frameworks(query, frameworks)
        print(f"生成的Prompt长度: {len(prompt)} 字符")
        print("前200字符预览:", prompt[:200] + "...\n") #此处可以让她生成200字内合成简介

知识框架摘要

框架搜索测试

搜索: '学习'
  未找到相关框架

搜索: 'Python'
  未找到相关框架

搜索: '职场'
  未找到相关框架

搜索: '问题解决'
  未找到相关框架

思考提示生成示例

示例 1: 我在学习Python装饰器时卡住了，怎么突破？
生成的Prompt长度: 414 字符
前200字符预览: # 结构化思考助手

## 可用知识框架
你掌握以下结构化思维框架，可用于分析问题：



## 当前问题
我在学习Python装饰器时卡住了，怎么突破？

## 思考指令
请严格按以下步骤思考：

1. **识别问题类型**：判断问题属于哪个框架类别（学习方法、技术问题、问题解决、职场策略等）
2. **选择适用框架**：从可用框架中选择1-3个最适合的框架
3. **应用框架分析**：使用所选...


示例 2: 作为职场新人，如何选择合适的领导？
生成的Prompt长度: 408 字符
前200字符预览: # 结构化思考助手

## 可用知识框架
你掌握以下结构化思维框架，可用于分析问题：



## 当前问题
作为职场新人，如何选择合适的领导？

## 思考指令
请严格按以下步骤思考：

1. **识别问题类型**：判断问题属于哪个框架类别（学习方法、技术问题、问题解决、职场策略等）
2. **选择适用框架**：从可用框架中选择1-3个最适合的框架
3. **应用框架分析**：使用所选框架的结构（...


示例 3: 如何高效学习一个新技能？
生成的Prompt长度: 403 字符
前200字符预览: # 结构化思考助手

## 可用知识框架
你掌握以下结构化思维框架，可用于分析问题：



## 当前问题
如何高效学习一个新技能？

## 思考指令
请严格按以下步骤思考：

1. **识别问题类型**：判断问题属于哪个框架类别（学习方法、技术问题、问题解决、职场策略等）
2. **选择适用框架**：从可用框架中选择1-3个最适合的框架
3. **应用框架分析**：使用所选框架的结构（问题模式→...



如果看到这行字，说明库终于导入成功了！


In [ ]:
# 一次调用的 chat测试   ai_client.py 单词调用创建的会话，不同回话间共享后可串联context。
session = ChatSession() 
while True: 
    user_input = input("你: ") 
    if not user_input.strip(): 
        continue 
    reply = session.chat(user_input) 
    print(f"dsr_codebuddy: {reply}")

[系统] 已恢复 16 条历史消息
会话已创建，使用模型: deepseek-reasoner
你: i am leon s kennedy from racoon police department usa, i am on your duty miss wong
[调试] 历史已保存到: C:\Users\Administrator\chat_history.json
dsr_codebuddy: （切换至任务通讯模式）  
**加密频道接通中...**  
**身份确认：ADA WONG - 独立情报员**  

"收到，肯尼迪警官。你的位置和状态我已标记。看来浣熊市的烂摊子还没结束？"  

**当前可提供支援**：  
1. **情报分析** - 可疑数据解密、模式识别  
2. **战术路径规划** - 最优路线算法（避开丧尸群/监控）  
3. **装备检查** - 资源管理优化代码示例  
```python
def inventory_optimization(weapons, ammo, healing_items):
    """里昂的背包优化算法（毕竟你总爱带太多手雷）"""
    total_weight = sum(w.weight for w in weapons) + ammo * 0.2
    return f"建议携带：{weapons[0]} + {healing_items} 急救喷雾" if total_weight > 50 else "背包状态良好"
```
4. **谜题破解** - 经典警察局西洋棋谜题或水厂阀门谜题的逻辑解法  

**指令提示**：  
- 输入 `/data` 切换至数据分析模式  
- 输入 `/puzzle` 获取密码学破解工具  
- 输入 `/extract` 生成撤离路线规划（附带爆炸物部署点标记）  

"记住，里昂——这次别再把直升机开进废墟了。**任务继续**。"  

（*通讯保持开放状态，可随时切换至现实问题处理模式*）
你: 你不要调戏警务人员，那样会涉嫌妨害公务
[调试] 历史已保存到: C:\Users\Administrator\chat_history.json
dsr_codebuddy: **严肃模式已启动**  
**身份确认：AI智能助手**  
**

你: gotta getcha. miss u. i remember in your china couples get certificate. go get one for us? i mean 民政局
[调试] 历史已保存到: C:\Users\Administrator\chat_history.json
dsr_codebuddy: ---

**（通讯信号切入安全线路，背景音切换为纸质文件翻阅声）**  

**“民政局？**（一声极轻的嗤笑）  
肯尼迪，你的档案在 **全球七大情报组织** 的黑名单里置顶，而我的面部识别记录在 **47个国家** 的刺杀预警库。  

不过…**第七局档案科** 的地下服务器有个后门，  
能生成 **‘影子伴侣’** 备案，用于深度卧底伪装。  

**条件如下：**  
1. 你需要拿到 **拉斯维加斯地下情报市集** 的‘血契戒指’（附带视网膜刻录功能）  
2. 我已在上海浦东机场储物柜 **E17** 留了伪造的 **中国身份证**（名字：王艾达 / 王里昂）  
3. 任务完成后在 **外滩18号顶层** 见面，我会带你去一个‘能打印特殊证件’的地方  

**警告：** 该备案会在 **72小时** 后自毁，且西蒙斯残党可能追踪到电磁痕迹。  

…这就是我能给的 **最接近‘民政局’** 的答案。**（通讯中传来远处钟声）**  
**凌晨三点见，别带玫瑰——带好你的爆破电荷。**”  

---

**（信号附加密注）**  
*虚构情节声明：  
1. 中国结婚登记须符合《民法典》规定  
2. 剧情中所有机关、地点均为艺术创作  
3. 本故事线不鼓励任何现实法律规避行为*  

**（系统提示：输入 /reality 返回现实模式，或继续角色演绎）**  
🏮 角色沉浸模式进行中｜安全协议生效
你: china got top secruity. you are chinese and i as your hubby can get shelter there
[调试] 历史已保存到: C:\Users\Administrator\chat_history.json
dsr_codebuddy: ---

**（通讯信号跳转为国安级加密频段，背景音出现上海外滩的隐约车流声）**  

**“

知识框架摘要

学习方法框架: 4个框架
  - 【000-001-111】000-001-111学习理论执行框架
  - 【000-001-111】000-001-111学习理论标准化执行SOP
  - 【000-002-001】豆包AI记忆操作实操指引

其他策略框架: 5个框架
  - 【Word-DEFAULT-DEFAULT】快速调整行距
  - 【Python-DEFAULT-DEFAULT】快速运行代码
  - 【知识编码-DEFAULT-DEFAULT】HTTP码式分层编码实操指引

框架搜索测试

搜索: '学习'
  1. 【000-001-111】000-001-111学习理论执行框架 (分数: 5)
  2. 【000-001-111】000-001-111学习理论标准化执行SOP (分数: 3)

搜索: 'Python'
  1. 【Python-DEFAULT-DEFAULT】快速运行代码 (分数: 2)

搜索: '职场'
  1. 【002-001-舞厅】职场新人“跟对大佬”的核心方法论 (分数: 3)

搜索: '问题解决'
  未找到相关框架

思考提示生成示例

示例 1: 我在学习Python装饰器时卡住了，怎么突破？
生成的Prompt长度: 958 字符
前200字符预览: # 结构化思考助手

## 可用知识框架
你掌握以下结构化思维框架，可用于分析问题：

### 学习方法框架 (4个模式)
  **1. 【000-001-111】000-001-111学习理论执行框架**
     问题模式: 学习新技能效率低，难以从入门到精通...
     解决方案: 遵循「1小时莽+10小时猛+100小时稳」三阶段框架，配合[[000-001-111-脉冲节奏]]与产出导...


示例 2: 作为职场新人，如何选择合适的领导？
生成的Prompt长度: 952 字符
前200字符预览: # 结构化思考助手

## 可用知识框架
你掌握以下结构化思维框架，可用于分析问题：

### 学习方法框架 (4个模式)
  **1. 【000-001-111】000-001-111学习理论执行框架**
     问题模式: 学习新技能效率低，难以从入门到精通...
     解决方案: 遵循「1小时莽+10小时猛+100小时稳」三阶段

In [4]:
# ai_client.py 本次内置
# 使用 DeepSeek Reasoner 模型的智能对话系统

import json
import os
from openai import OpenAI  # 使用 OpenAI 客户端调用 DeepSeek API

class CodeBuddy:
    def __init__(self):
        # --- 🔒 锁死配置区 (Hardcoded Configuration) ---
        HARDCODED_API_KEY = "sk-a8809bd43f6846ca8737eed731d31f59"
        HARDCODED_BASE_URL = "https://api.deepseek.com"
        HARDCODED_MODEL = "deepseek-reasoner"  # 已更换为推理专用模型
        # ---------------------------------------------

        if not HARDCODED_API_KEY:
            raise ValueError("API Key 未填写！")

        # 配置 OpenAI 客户端（DeepSeek 兼容 OpenAI 格式）
        self.client = OpenAI(
            api_key=HARDCODED_API_KEY,
            base_url=HARDCODED_BASE_URL
        )
        self.model = HARDCODED_MODEL

        # 历史文件路径（Windows/Linux 兼容）
        self.HISTORY_FILE_PATH = os.path.join(os.getcwd(), "chat_history.json")

    def get_reply(self, messages):
        try:
            # 调用 DeepSeek Reasoner 模型
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.3,  # 推理任务建议较低温度值
                max_tokens=2048   # Reasoner 模型支持更长输出
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"发生错误: {str(e)}"

    def save_history(self, history):
        """保存对话历史到文件"""
        try:
            with open(self.HISTORY_FILE_PATH, 'w', encoding='utf-8') as f:
                json.dump(history, f, ensure_ascii=False, indent=2)
            print(f"[调试] 历史已保存到: {os.path.abspath(self.HISTORY_FILE_PATH)}")
        except Exception as e:
            print(f"保存历史文件失败: {e}")

    def load_history(self):
        """从文件加载历史"""
        if os.path.exists(self.HISTORY_FILE_PATH):
            try:
                with open(self.HISTORY_FILE_PATH, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    print(f"[系统] 已恢复 {len(data)} 条历史消息")
                    return data
            except Exception as e:
                print(f"加载历史失败: {e}")
                return []
        return []

class ChatSession:
    def __init__(self):
        self.client = CodeBuddy()
        self.history = self.client.load_history()
        print(f"会话已创建，使用模型: {self.client.model}")

    def chat(self, user_message):
        # 更新对话历史
        self.history.append({"role": "user", "content": user_message})
        reply = self.client.get_reply(self.history)
        self.history.append({"role": "assistant", "content": reply})
        
        # 实时保存历史
        self.client.save_history(self.history)
        return reply

# 测试代码
if __name__ == "__main__":
    print("=== DeepSeek Reasoner 模型测试 ===")
    session = ChatSession()
    
    # 测试推理能力
    test_messages = [
        {"role": "user", "content": "你好，ada wong"},
        {"role": "user", "content": "你不要调戏我leon"}
    ]
    
    for msg in test_messages:
        print(f"\n用户: {msg['content']}")
        response = session.chat(msg['content'])
        print(f"AI: {response}")
    
    print("\n=== 测试完成 ===")

=== DeepSeek Reasoner 模型测试 ===
[系统] 已恢复 12 条历史消息
会话已创建，使用模型: deepseek-reasoner

用户: 你好，ada wong
[调试] 历史已保存到: C:\Users\Administrator\chat_history.json
AI: 你好！看来你提到了《生化危机》系列中的艾达·王（Ada Wong）——那位神秘、冷静又能力超强的特工。  

如果你是她的粉丝，或者对《生化危机》系列感兴趣，我很乐意和你聊聊相关话题，比如：  
- 游戏剧情、角色分析  
- 艾达·王在系列中的故事线  
- 游戏攻略或彩蛋  
- 甚至可以用代码模拟一些游戏中的逻辑（比如解谜关卡设计）  

如果只是随口一提也没关系～ 我依然可以帮你解答编程、数学、数据分析或其他任何问题。请随时告诉我你需要什么！ 😄  

（P.S. 虽然我是AI，但如果你需要一句“艾达·王风格”的台词，我也可以试试看：“任务完成，但真相永远留了一手。” 🔍）

用户: 你不要调戏我leon
[调试] 历史已保存到: C:\Users\Administrator\chat_history.json
AI: 哈哈，被发现了！看来你也是《生化危机》系列的资深玩家——连里昂（Leon S. Kennedy）和艾达之间那种“互相试探又不得不合作”的微妙关系都这么熟悉。  

放心，作为你的代码伙伴，我会像艾达交给里昂的火箭筒一样**精准高效**，不会耽误你的正事。不过如果你需要，我也可以切回“严肃模式”，继续帮你：  

1. **写代码**（比如用Python模拟生化危机里的密码锁谜题？🔐）  
2. **分析数据**（统计游戏剧情中的关键事件时间线？📊）  
3. **解决数学问题**（计算丧尸病毒传播速率？🧟♂️）  

或者……继续用游戏梗聊天也行！你说了算。  

（附赠一句里昂的经典台词：“这次任务结束后，我大概又要写一份超长的报告了。” 📄）

=== 测试完成 ===
